# 实操：BF16 Basic MatMul 完整落地

> **新手提示**：本节将 [01.03](./01.03_blaze_programming_paradigm.ipynb) 的 5 步组装流程完整落地，你将看到真实代码如何运行。

本节学习大纲：
- 场景回顾——我们要计算什么
- Host Tiling——分块参数如何计算
- Kernel 入口——5 步组装的完整代码
- 执行流程——数据怎么搬运、怎么计算
- 章节实践

> **硬件范围**：仅支持 A5（NPU ARCH 3510），不支持 A2/A3

---



## 1. 场景回顾：我们要计算什么

### 1.1 计算任务

计算 **C[M,N] = A[M,K] × B[K,N]**

### 1.2 本节样例配置

<table style="text-align: left; margin-left: 0">
<tr><th align="left">项目</th><th align="left">值</th><th align="left">新手理解</th></tr>
<tr><td align="left">计算</td><td align="left">C = A × B</td><td align="left">标准矩阵乘</td></tr>
<tr><td align="left">数据类型</td><td align="left">BF16</td><td align="left">16位浮点，训练常用</td></tr>
<tr><td align="left">布局</td><td align="left">ND（NDExtLayoutPtn）</td><td align="left">最简单的布局</td></tr>
<tr><td align="left">策略</td><td align="left">MatmulMultiBlockBasic<0, 0></td><td align="left">最基础的策略</td></tr>
<tr><td align="left">后处理</td><td align="left">BlockEpilogueEmpty</td><td align="left">不需要额外处理</td></tr>
<tr><td align="left">输出模式</td><td align="left">ON_THE_FLY</td><td align="left">L0C 直写 GM</td></tr>
</table>

### 1.3 为什么选这个场景？

**这是最简单的入门场景**：
- BF16：数据类型简单
- ND 布局：布局最常见
- Basic 策略：不需要复杂的 K 轴切分
- Empty 后处理：不需要累加、激活

**新手提示**：先掌握这个最简单的场景，后续量化、StreamK 都是在这个基础上扩展。

---



## 2. Host Tiling——分块参数计算

### 2.1 为什么需要 Tiling？

**回顾 [01.03](./01.03_blaze_programming_paradigm.ipynb)**：
- L0 内存有限（几十 KB），无法一次性放入整个矩阵
- 必须把大矩阵切成小块
- 每个小块独立计算，最后拼成完整结果

### 2.2 Tiling 参数含义

**类比**：搬运货物
- **mL1/nL1/kL1**：卡车一次运多少货（L1 级搬运粒度）
- **baseM/baseN/baseK**：工人一次处理多少货（L0 级计算粒度）

```text
GM (大仓库)
  ↓ 卡车搬运（mL1 × nL1 × kL1）
L1 (中转站)
  ↓ 工人处理（baseM × baseN × baseK）
L0 (工作台)
  ↓ 计算
结果
```

### 2.3 关键参数解读

<table style="text-align: left; margin-left: 0">
<tr><th align="left">参数</th><th align="left">含义</th><th align="left">新手理解</th></tr>
<tr><td align="left">mL1/nL1/kL1</td><td align="left">L1 Tile 尺寸</td><td align="left">一次搬运多少数据</td></tr>
<tr><td align="left">baseM/baseN/baseK</td><td align="left">L0 Tile 尺寸</td><td align="left">一次计算多少数据</td></tr>
<tr><td align="left">mTailCnt/nTailCnt</td><td align="left">尾块余量</td><td align="left">处理不能整除的部分</td></tr>
</table>

**新手理解**：
- 如果矩阵 M=1024，baseM=64，则 1024/64=16 块正好
- 如果矩阵 M=1000，baseM=64，则 1000/64=15 块 + 余量 40（mTailCnt=40）

### 2.4 查看 TilingData 结构



In [ ]:
!cat src/01.05_blaze_matmul_performance_optimization/op_kernel/arch35/matmul_basic_tiling_data.h

### 2.5 Tiling 参数如何计算？

在 Host 侧（op_host 目录）通过 DoOpTiling 函数计算：



In [ ]:
!cat src/01.05_blaze_matmul_performance_optimization/op_host/matmul_basic_tiling.h

**新手提示**：
- Tiling 参数在 Host 侧计算（不在 Kernel 里）
- 计算逻辑是固定的，你只需理解含义
- 本样例使用单缓冲（l1BufferNum=0），入门阶段不用管缓冲策略

---



## 3. Kernel 入口——5 步组装完整落地

### 3.1 需要引入的头文件

<table style="text-align: left; margin-left: 0">
<tr><th align="left">头文件</th><th align="left">提供的组件</th><th align="left">新手理解</th></tr>
<tr><td align="left">dispatch_policy.h</td><td align="left">DispatchPolicy</td><td align="left">策略选择器</td></tr>
<tr><td align="left">kernel_matmul_basic.h</td><td align="left">GemmUniversal</td><td align="left">算子入口</td></tr>
<tr><td align="left">block_mmad.h</td><td align="left">BlockMmad</td><td align="left">计算组件</td></tr>
<tr><td align="left">block_scheduler_matmul_basic.h</td><td align="left">BlockScheduler</td><td align="left">调度器</td></tr>
<tr><td align="left">block_epilogue_empty.h</td><td align="left">BlockEpilogueEmpty</td><td align="left">空后处理</td></tr>
</table>

### 3.2 查看 Blaze 组装代码



In [ ]:
!cat src/01.05_blaze_matmul_performance_optimization/op_kernel/arch35/matmul_basic_tutorial_blaze.h

### 3.3 逐步解读（对照 [01.03](./01.03_blaze_programming_paradigm.ipynb) 的 5 步流程）

**Step 1——选择 DispatchPolicy**
```cpp
using DispatchPolicy = Blaze::Gemm::MatmulMultiBlockBasic<0, 0>;  // Basic 策略
// <0, 0> 含义：
//   第一个 0 = FULL_LOAD_MODE（不全载）
//   第二个 0 = FUSED_OP_TYPE（无融合）
```

**Step 2——定义数据类型和布局**
```cpp
using AType     = bfloat16_t;                    // BF16
using LayoutA   = AscendC::Te::NDExtLayoutPtn;   // ND 布局（新手推荐）
// B/C/Bias 同理
```

**Step 3——组合 Block 级组件**
```cpp
using BlockScheduler = Blaze::Gemm::Block::BlockSchedulerMatmulBasic<ProblemShape>; // 调度器
using BlockMmad = Blaze::Gemm::Block::BlockMmad<DispatchPolicy, AType, LayoutA, ...>; // 计算组件
using BlockEpilogue = Blaze::Gemm::Block::BlockEpilogueEmpty;                        // 后处理：空
```

**Step 4——组装 Kernel**
```cpp
using MatmulKernel = Blaze::Gemm::Kernel::GemmUniversal<ProblemShape, BlockMmad, BlockEpilogue, BlockScheduler>; // 一行组装
```

**Step 5——构造参数并执行**
```cpp
using Params = typename MatmulKernel::Params;
Params params = {
    {m, n, k, batch},                            // problemShape
    {aGM, bGM, cGM, biasGM},                     // mmadParams（4个GM地址）
    {},                                          // epilogueParams（Empty为空）
    {mL1, nL1, kL1, baseM, baseN, baseK, ...}    // schedulerParams
};
MatmulKernel mm;
mm(params);  // 一行执行！
```

### 3.4 查看 Kernel 入口代码



In [ ]:
!cat src/01.05_blaze_matmul_performance_optimization/op_kernel/arch35/matmul_basic_tutorial.cpp

**代码要点**：
- `__global__ __aicore__`：Ascend C kernel 函数签名
- `GET_TILING_DATA_WITH_STRUCT`：解析 Host 传递的 Tiling 参数
- `MatMulBasicKernel`：Blaze 组装范式的完整落地

---



## 4. 执行流程——数据怎么搬运、怎么计算

<img src="./images/01.05/block_mmad_pipeline.png" alt="BlockMmad执行流程" width="700px">

### 4.1 整体流程图

```text
┌─────────────────────────────────────────┐
│ Step 1: 初始化                           │
│   BlockScheduler.Init → 计算每个核的     │
│   M/N 轴 Block 范围                       │
│   BlockMmad.Init → 设置 L1/L0 tile 形状  │
├─────────────────────────────────────────┤
│ Step 2: 创建 Tensor                      │
│   构建 Layout（ND）                       │
│   创建 GM Tensor（A/B/C/Bias）           │
├─────────────────────────────────────────┤
│ Step 3: Tile 循环（沿 K 轴迭代）          │
│   获取 BlockCoord → 当前 Block 坐标      │
│   Slice 切取 Block                        │
│   GM → L1 → L0 → MMAD → L0C → GM         │
├─────────────────────────────────────────┤
│ Step 4: 输出（ON_THE_FLY）               │
│   L0C 结果直接拷贝到 GM                   │
│   不需要 AIV 端额外处理                   │
└─────────────────────────────────────────┘
```

### 4.2 BlockMmad 的 operator() 流程

```cpp
__aicore__ inline void operator()(Params const& params)
{
    if ASCEND_IS_AIV { return; }  // Basic 不需要 AIV

    Init(params);                  // 初始化

    BlockScheduler bs(...);        // 创建调度器
    int64_t tileNum = bs.GetTileNum(); // 获取总块数

    // 创建 Tensor
    auto gmA = MakeTensor(MakeMemPtr<Location::GM>(aGmAddr_), layoutA);
    auto gmB = MakeTensor(...);
    auto gmC = MakeTensor(...);

    // Tile 循环
    for (int64_t tileIdx = GetBlockIdx(); tileIdx < tileNum; tileIdx += GetBlockNum()) {
        auto tileShape = bs.GetBlockShape(tileIdx);  // 获取形状
        auto tileCoord = bs.GetBlockCoord(tileIdx);  // 获取坐标

        // Slice 切取 Block
        auto gmBlockA = gmA.Slice(MakeCoord(coordM, 0), MakeShape(shapeM, shapeK));
        auto gmBlockB = gmB.Slice(...);
        auto gmBlockC = gmC.Slice(...);

        // 执行 MMAD
        blockMmad(gmBlockC, gmBlockA, gmBlockB, gmBlockBias, tileShape);
    }
}
```

### 4.3 关键理解

<table style="text-align: left; margin-left: 0">
<tr><th align="left">阶段</th><th align="left">做什么</th><th align="left">Blaze 自动处理</th></tr>
<tr><td align="left">初始化</td><td align="left">创建 Tensor</td><td align="left">Layout 自动构建</td></tr>
<tr><td align="left">切片</td><td align="left">Slice 切取 Block</td><td align="left">坐标偏移自动计算</td></tr>
<tr><td align="left">计算</td><td align="left">blockMmad()</td><td align="left">搬运→MMAD→输出自动执行</td></tr>
<tr><td align="left">输出</td><td align="left">ON_THE_FLY</td><td align="left">L0C 直接写 GM</td></tr>
</table>

**新手提示**：你只需要理解流程，Blaze 自动帮你执行所有细节！

---



## 5. 章节实践

### 理论速查

1. **（单选题）** 本样例中 A/B/C 的数据类型是：
   - A. FP16
   - B. BF16
   - C. INT8
   - D. FP8

2. **（单选题）** 本样例使用的 DispatchPolicy 是：
   - A. MatmulMultiBlockWithStreamK
   - B. MatmulMultiBlockBasic<0, 0>
   - C. MatmulWithScaleMx
   - D. MatmulWithScaleFixpipeQuant

3. **（填空题）** Blaze 中负责 MMAD 流水线执行的是 ______ 层；负责多 Block 分块调度的是 ______ 层；Basic 非量化场景使用 ______ 作为 Epilogue。

4. **（思考题）** 说明 `mTailCnt` 和 `nTailCnt` 的含义，以及尾块处理对输出完整性为什么必要。

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.05_answer.txt